# Predicción reproducible de supervivencia

Ejecuta `make pipeline` primero para generar Silver. El conjunto de prueba se reserva desde el inicio y solo se usa en la celda final de evaluación. Las decisiones son exploratorias: no deben trasladarse a decisiones actuales sobre personas.

In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    brier_score_loss,
    classification_report,
    f1_score,
    precision_recall_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    learning_curve,
)

from flows.common.config import settings
from flows.common.hashing import sha256_of
from flows.common.paths import BRONZE, BRONZE_UNLABELED_NAME, SILVER, SILVER_NAME

ml = importlib.import_module("flows.05_ml_models")
X, y = ml.load_dataset()
X_train, X_test, y_train, y_test = ml.split(X, y)
print("Objetivo y proporción:", y.name, y.mean(), "; entrenamiento/prueba:", X_train.shape, X_test.shape)
print("Tasa supervivencia train/test:", y_train.mean(), y_test.mean(), "(estratificación)")

## Pasos 39–43: problema, fuga y baselines

La unidad es un pasajero al embarcar; `Survived` es la etiqueta. Se prioriza ROC-AUC por no depender de un umbral. La edad de Silver fue imputada usando todos los pasajeros; el modelo recibe `AgeOriginal` y aprende su imputación dentro de cada pliegue.

In [ ]:
all_female_mean = X.loc[X["Sex"] == "female", "Age"].mean()
train_female_mean = X_train.loc[X_train["Sex"] == "female", "Age"].mean()
print("Promedio global (referencia con fuga):", all_female_mean)
print("Promedio femenino de entrenamiento:", train_female_mean)
folds = StratifiedKFold(n_splits=settings.cv_folds, shuffle=True, random_state=settings.random_state)
dummy_oof = cross_val_predict(DummyClassifier(strategy="most_frequent"), X_train, y_train, cv=folds)
gender_oof = (X_train["Sex"].to_numpy() == "female").astype(int)
print("Baseline mayoría accuracy:", accuracy_score(y_train, dummy_oof))
print("Regla mujer->sobrevive:", classification_report(y_train, gender_oof, zero_division=0))
cv_results = ml.cross_validate_candidates(X_train, y_train)
display(cv_results)

## Pasos 44–50: imputación, pipeline y selección

`GroupAgeImputer` aprende promedios solo con entrenamiento; las categóricas se codifican con one-hot y las variables numéricas se escalan. Compara media, desviación y brecha train-validación para elegir finalistas.

In [ ]:
pipe = ml.build_pipeline(ml.get_candidates(settings.random_state)["logreg"])
pre = pipe[:-1].fit(X_train, y_train)
Z = pre.transform(X_test)
feature_names = pre.named_steps["prep"].get_feature_names_out()
print("NaN tras preprocesamiento:", np.isnan(Z).any(), "; columnas:", Z.shape[1])
print([name for name in feature_names if "Sex_" in name])
cv_results["train_valid_gap"] = cv_results["train_roc_auc_mean"] - cv_results["test_roc_auc_mean"]
display(cv_results[["model", "test_roc_auc_mean", "test_roc_auc_std", "train_valid_gap"]])
grid = GridSearchCV(
    ml.build_pipeline(ml.get_candidates(settings.random_state)["hgb"]),
    ml.get_param_grids()["hgb"], scoring=ml.PRIMARY_METRIC, cv=folds,
    n_jobs=1, refit=True, error_score="raise",
)
grid.fit(X_train, y_train)
print("Mejores parámetros CV:", grid.best_params_, "ROC-AUC:", grid.best_score_)
display(pd.DataFrame(grid.cv_results_).sort_values("rank_test_score").head(10))
default_hgb = ml.build_pipeline(ml.get_candidates(settings.random_state)["hgb"])
default_scores = cross_validate(default_hgb, X_train, y_train, cv=folds, scoring="roc_auc", error_score="raise")["test_score"]
inner_folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=settings.random_state + 1)
nested_search = GridSearchCV(
    ml.build_pipeline(ml.get_candidates(settings.random_state)["hgb"]),
    ml.get_param_grids()["hgb"], scoring=ml.PRIMARY_METRIC,
    cv=inner_folds, n_jobs=1, refit=True, error_score="raise",
)
nested_scores = cross_validate(nested_search, X_train, y_train, cv=folds, scoring="roc_auc", error_score="raise")["test_score"]
paired_differences = nested_scores - default_scores
print("Nested CV ROC-AUC:", nested_scores.mean(), "+/-", nested_scores.std())
print("Diferencia pareada nested - default:", paired_differences.mean(), "+/-", paired_differences.std())

## Pasos 51–55: errores, umbral y estabilidad

Las predicciones OOF de entrenamiento permiten revisar subgrupos sin consultar el conjunto de prueba. Se calcula el umbral que maximiza F1 como referencia, pero se conserva 0.5 porque no hay una función de costos aprobada para convertir un umbral en política.

In [ ]:
selected = ml.build_pipeline(ml.get_candidates(settings.random_state)["hgb"])
selected.set_params(**ml.FINAL_PARAMS)
oof = cross_val_predict(selected, X_train, y_train, cv=folds, method="predict_proba")[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, oof)
f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_oof_threshold = float(thresholds[np.argmax(f1_values)])
threshold = ml.THRESHOLD
oof_pred = (oof >= threshold).astype(int)
print("Umbral que maximiza F1 en OOF:", best_oof_threshold, "; umbral congelado:", threshold, "; Brier:", brier_score_loss(y_train, oof))
print(classification_report(y_train, oof_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_train, oof_pred); plt.show()
RocCurveDisplay.from_predictions(y_train, oof); plt.show()
PrecisionRecallDisplay.from_predictions(y_train, oof); plt.show()
prob_true, prob_pred = calibration_curve(y_train, oof, n_bins=8)
plt.plot(prob_pred, prob_true, marker="o"); plt.plot([0, 1], [0, 1], "--")
plt.xlabel("Probabilidad predicha"); plt.ylabel("Frecuencia observada"); plt.title("Calibración OOF"); plt.show()
errors = X_train.copy(); errors["y"] = y_train; errors["pred"] = oof_pred; errors["error"] = errors["y"] != errors["pred"]
errors["Person"] = np.where(errors["Age"] < 16, "child", errors["Sex"])
for group in ["Sex", "Pclass", "Person"]:
    summary = errors.groupby(group).agg(n=("error", "size"), accuracy=("error", lambda s: 1 - s.mean()))
    summary["recall"] = errors.groupby(group).apply(lambda g: ((g["pred"] == 1) & (g["y"] == 1)).sum() / g["y"].sum() if g["y"].sum() else np.nan)
    display(summary)
confidence = np.where(oof_pred == 1, oof, 1 - oof)
error_positions = np.flatnonzero(oof_pred != y_train.to_numpy())
hard_errors = error_positions[np.argsort(-confidence[error_positions])[:10]]
display(X_train.iloc[hard_errors].assign(y=y_train.iloc[hard_errors].to_numpy(), pred=oof_pred[hard_errors], proba=oof[hard_errors]))
sizes, train_scores, valid_scores = learning_curve(selected, X_train, y_train, cv=folds, train_sizes=np.linspace(.2, 1, 5), scoring="roc_auc", n_jobs=1)
plt.plot(sizes, train_scores.mean(axis=1), label="train"); plt.plot(sizes, valid_scores.mean(axis=1), label="validación")
plt.legend(); plt.xlabel("Tamaño de entrenamiento"); plt.ylabel("ROC-AUC"); plt.show()

## Pasos 56–61: ajuste e interpretación

La búsqueda se hace exclusivamente en entrenamiento. La importancia por permutación usa un subconjunto de validación del entrenamiento; las ablaciones comparan las mismas particiones. La cabina puede reflejar sesgo de registro, no protección causal.

In [ ]:
from sklearn.model_selection import train_test_split

X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=.25, random_state=settings.random_state, stratify=y_train)
interpretable = ml.build_pipeline(ml.get_candidates(settings.random_state)["logreg"]).fit(X_fit, y_fit)
permutation = permutation_importance(interpretable, X_val, y_val, scoring="roc_auc", n_repeats=30, random_state=settings.random_state)
display(pd.DataFrame({"feature": X_val.columns, "importance": permutation.importances_mean, "std": permutation.importances_std}).sort_values("importance", ascending=False))
coef = pd.Series(interpretable.named_steps["model"].coef_[0], index=interpretable.named_steps["prep"].get_feature_names_out())
display(pd.DataFrame({"coef": coef, "odds_ratio": np.exp(coef)}).assign(abs_coef=lambda t: t.coef.abs()).sort_values("abs_coef", ascending=False).drop(columns="abs_coef").head(15))
ablations = {
    "base": (ml.NUM_FEATURES, ml.CAT_FEATURES),
    "base+cabina": (ml.NUM_FEATURES + ml.CABIN_NUM, ml.CAT_FEATURES + ml.CABIN_CAT),
    "sin_titulo": (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != "TTitulo"]),
    "sin_sexo": (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != "Sex"]),
}
ablation_results = []
for name, (numeric, categorical) in ablations.items():
    candidate = ml.build_pipeline(ml.get_candidates(settings.random_state)["logreg"], numeric, categorical)
    scores = cross_validate(candidate, X_train, y_train, cv=folds, scoring="roc_auc", error_score="raise")["test_score"]
    ablation_results.append({"configuración": name, "ROC-AUC": scores.mean(), "desviación": scores.std()})
display(pd.DataFrame(ablation_results))

## Pasos 62–69: evaluación final y artefacto

Esta es la única celda que evalúa sobre `X_test`. Reporta intervalo bootstrap, luego reentrena con todos los pasajeros etiquetados, guarda metadatos y comprueba la inferencia en Bronze sin etiqueta.

In [ ]:
evaluation_pipeline = ml.build_pipeline(ml.get_candidates(settings.random_state)[ml.FINAL_ESTIMATOR])
evaluation_pipeline.set_params(**ml.FINAL_PARAMS).fit(X_train, y_train)
test_proba = evaluation_pipeline.predict_proba(X_test)[:, 1]
test_metrics = ml.metrics_from_probabilities(y_test, test_proba, threshold)
test_metrics["roc_auc_ci_95"] = ml._bootstrap_roc_auc_interval(y_test, test_proba, settings.random_state)
test_dummy = np.full(len(y_test), y_train.mode().iloc[0])
test_gender = (X_test["Sex"].to_numpy() == "female").astype(int)
test_comparison = pd.DataFrame([
    {"modelo": "majority", "accuracy": accuracy_score(y_test, test_dummy), "f1": f1_score(y_test, test_dummy, zero_division=0)},
    {"modelo": "regla género", "accuracy": accuracy_score(y_test, test_gender), "f1": f1_score(y_test, test_gender, zero_division=0)},
    {"modelo": ml.FINAL_ESTIMATOR, "accuracy": test_metrics["accuracy"], "f1": test_metrics["f1"]},
])
display(test_comparison)
display(test_metrics)
final_model = ml.train_final(X, y)
metadata = {
    "estimator": ml.FINAL_ESTIMATOR, "params": ml.FINAL_PARAMS,
    "features": {"numeric": ml.NUM_FEATURES, "categorical": ml.CAT_FEATURES},
    "target": ml.TARGET, "threshold": threshold, "random_state": settings.random_state,
    "n_train": len(X), "data_sha256": sha256_of(SILVER / SILVER_NAME),
    "metrics": test_metrics,
}
model_path = ml.save_model(final_model, metadata)
loaded, card = ml.load_model(model_path)
print("Guardado:", model_path, "; versión sklearn:", card["sklearn_version"])
unlabeled = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
if len(unlabeled):
    engineering = importlib.import_module("flows.01_data_engineering")
    unlabeled["Survived"] = 0
    prepared = engineering.build_silver(unlabeled)
    X_new = prepared[ml.CANDIDATE_COLUMNS].copy()
    X_new["Age"] = prepared["AgeOriginal"]
    smoke = ml.predict(loaded, X_new, threshold=threshold)
    display(smoke.describe())
    assert not smoke.isna().any().any()